# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/minamalak123/FlyRankIntern/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:


import os
import numpy as np
import pandas as pd
import duckdb

from pathlib import Path
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download

print("Libraries loaded.")

c:\Users\minam\OneDrive\Desktop\INTERN\FlyRankIntern\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded.


In [5]:


env_path = Path.cwd().parents[1] / ".env"

print("Looking for:", env_path)
print("Exists:", env_path.exists())

load_dotenv(env_path)

HF_TOKEN = os.getenv("HF_TOKEN")

assert HF_TOKEN, "HF_TOKEN was not found in .env"

print("HF token loaded successfully")

Looking for: c:\Users\minam\OneDrive\Desktop\INTERN\FlyRankIntern\.env
Exists: True
HF token loaded successfully


In [6]:


con = duckdb.connect()

con.execute("""
INSTALL https;
LOAD https;
""")

print("DuckDB connection ready")

DuckDB connection ready


In [7]:

feb_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-02/data_0.parquet",
    token=HF_TOKEN
)

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN
)

print("February file:", feb_file)
print("March file:", march_file)

c:\Users\minam\OneDrive\Desktop\INTERN\FlyRankIntern\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\minam\.cache\huggingface\hub\datasets--FlyRank--internship-warehouse. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


February file: C:\Users\minam\.cache\huggingface\hub\datasets--FlyRank--internship-warehouse\snapshots\50cbf7c3909d07be4d1b5906b4d09e882e5acbf2\fact_content_daily_performance\month=2026-02\data_0.parquet
March file: C:\Users\minam\.cache\huggingface\hub\datasets--FlyRank--internship-warehouse\snapshots\50cbf7c3909d07be4d1b5906b4d09e882e5acbf2\fact_content_daily_performance\month=2026-03\data_0.parquet


In [8]:

feb_features = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS impressions_feb,

    SUM(gsc_clicks) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS clicks_feb,

    CASE
        WHEN SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) > 0
        THEN
            SUM(
                gsc_impressions * gsc_avg_position
            ) FILTER (
                WHERE gsc_data_available IS TRUE
                  AND gsc_avg_position IS NOT NULL
            )
            /
            SUM(gsc_impressions) FILTER (
                WHERE gsc_data_available IS TRUE
            )
        ELSE NULL
    END AS avg_position_feb,

    SUM(ga4_engaged_sessions) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS engaged_sessions_feb,

    SUM(ga4_sessions) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS sessions_feb

FROM read_parquet('{feb_file}')

GROUP BY
    client_hash_id,
    content_hash_id
""").fetchdf()

feb_features["ctr_feb"] = (
    feb_features["clicks_feb"]
    / feb_features["impressions_feb"].replace(0, np.nan)
)

feb_features["engagement_rate_feb"] = (
    feb_features["engaged_sessions_feb"]
    / feb_features["sessions_feb"].replace(0, np.nan)
)

five_features = feb_features[
    [
        "client_hash_id",
        "content_hash_id",
        "impressions_feb",
        "clicks_feb",
        "ctr_feb",
        "avg_position_feb",
        "engagement_rate_feb"
    ]
].copy()

print("Five-feature frame created.")
print("Rows:", len(five_features))

five_features.head()

Five-feature frame created.
Rows: 321546


,client_hash_id,content_hash_id,impressions_feb,clicks_feb,ctr_feb,avg_position_feb,engagement_rate_feb
0,client_08a6a72ff48e62c0,content_f0c4713d2abc2238,NaN,NaN,NaN,NaN,NaN
1,client_08a6a72ff48e62c0,content_336421d79454e18f,22.0,0.0,0.0,48.318182,NaN
2,client_08a6a72ff48e62c0,content_d857ec403a88981b,3.0,0.0,0.0,71.666667,NaN
3,client_08a6a72ff48e62c0,content_94e42602fe4f2c19,1.0,0.0,0.0,6.000000,NaN
4,client_08a6a72ff48e62c0,content_f871b05188392cee,NaN,NaN,NaN,NaN,NaN


In [9]:


march_outcomes = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_clicks) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS clicks_mar,

    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS measured_gsc_days_mar

FROM read_parquet('{march_file}')

GROUP BY
    client_hash_id,
    content_hash_id
""").fetchdf()

march_outcomes = march_outcomes[
    march_outcomes["measured_gsc_days_mar"] > 0
].copy()

march_outcomes["clicks_mar"] = (
    march_outcomes["clicks_mar"].fillna(0)
)

march_outcomes["went_dark"] = (
    march_outcomes["clicks_mar"] == 0
).astype(int)

print("March outcome created.")
print("Rows:", len(march_outcomes))
print("Positive labels:", int(march_outcomes["went_dark"].sum()))

March outcome created.
Rows: 176738
Positive labels: 107901


In [10]:

model_frame = five_features.merge(
    march_outcomes[
        [
            "client_hash_id",
            "content_hash_id",
            "clicks_mar",
            "measured_gsc_days_mar",
            "went_dark"
        ]
    ],
    on=[
        "client_hash_id",
        "content_hash_id"
    ],
    how="inner"
)

print("model_frame created.")
print("Rows:", len(model_frame))
print("Columns:", list(model_frame.columns))

model_frame.head()

model_frame created.
Rows: 161539
Columns: ['client_hash_id', 'content_hash_id', 'impressions_feb', 'clicks_feb', 'ctr_feb', 'avg_position_feb', 'engagement_rate_feb', 'clicks_mar', 'measured_gsc_days_mar', 'went_dark']


,client_hash_id,content_hash_id,impressions_feb,clicks_feb,ctr_feb,avg_position_feb,engagement_rate_feb,clicks_mar,measured_gsc_days_mar,went_dark
0,client_08a6a72ff48e62c0,content_f0c4713d2abc2238,NaN,NaN,NaN,NaN,NaN,0.0,1,1
1,client_08a6a72ff48e62c0,content_336421d79454e18f,22.0,0.0,0.0,48.318182,NaN,0.0,11,1
2,client_08a6a72ff48e62c0,content_d857ec403a88981b,3.0,0.0,0.0,71.666667,NaN,0.0,20,1
3,client_08a6a72ff48e62c0,content_94e42602fe4f2c19,1.0,0.0,0.0,6.000000,NaN,0.0,1,1
4,client_08a6a72ff48e62c0,content_f871b05188392cee,NaN,NaN,NaN,NaN,NaN,0.0,7,1


## Method choice and why

My lane is **Refresh / Content Opportunity Scoring**.

The goal is to rank content items by their likelihood of experiencing the future `went_dark` outcome, so that limited review capacity can be focused on the highest-priority items.

I chose **Random Forest** as the primary model because it can capture nonlinear relationships and interactions between the February performance signals without requiring a linear relationship between each feature and the outcome.

The model will be evaluated against my Week-4 baseline using the same data, the same split, and the same ranking metric. Model complexity alone will not be treated as evidence of improvement.

I will also inspect feature importance and prediction errors so the result can be interpreted as a decision-support ranking rather than an unexplained model score.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Split design

I use a **client-grouped holdout split**.

The prediction unit is `client_hash_id × content_hash_id`, but the client identifier is not used as a predictive feature. Instead, `client_hash_id` is used only to create groups for validation.

This prevents the same client from appearing in both the training and test sets, making the evaluation more honest and reducing the risk of learning client-specific patterns.

I use a fixed random seed so the split is reproducible.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

print("Libraries loaded.")

Libraries loaded.


In [12]:
feature_cols = [
    "impressions_feb",
    "clicks_feb",
    "ctr_feb",
    "avg_position_feb",
    "engagement_rate_feb"
]

X = model_frame[feature_cols].copy()
y = model_frame["went_dark"].copy()
groups = model_frame["client_hash_id"].copy()

print("Features:")
for feature in feature_cols:
    print("-", feature)

print("\nX shape:", X.shape)
print("y shape:", y.shape)

Features:
- impressions_feb
- clicks_feb
- ctr_feb
- avg_position_feb
- engagement_rate_feb

X shape: (161539, 5)
y shape: (161539,)


In [13]:
print("Target distribution:")
print(y.value_counts())

print("\nTarget proportions:")
print(y.value_counts(normalize=True))

Target distribution:
went_dark
1    98368
0    63171
Name: count, dtype: int64

Target proportions:
went_dark
1    0.608943
0    0.391057
Name: proportion, dtype: float64


In [14]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx].copy()
groups_test = groups.iloc[test_idx].copy()

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training clients:", groups_train.nunique())
print("Test clients:", groups_test.nunique())

Training rows: 105652
Test rows: 55887
Training clients: 33
Test clients: 9


In [15]:
client_overlap = set(groups_train.unique()).intersection(
    set(groups_test.unique())
)

print("Client overlap:", len(client_overlap))

assert len(client_overlap) == 0

print("Grouped split check passed.")

Client overlap: 0
Grouped split check passed.


In [16]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_train_imp = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=feature_cols,
    index=X_train.index
)

X_test_imp = pd.DataFrame(
    imputer.transform(X_test),
    columns=feature_cols,
    index=X_test.index
)

print("Missing-value treatment complete.")

Missing-value treatment complete.


## Train + compare vs my baseline

I compare the learned models with my Week-4 baseline using the same held-out test set and the same evaluation metric.

Because this is a content prioritization problem, I focus on ranking performance rather than accuracy alone. I use Average Precision and Precision@K to measure how effectively the model places `went_dark` items near the top of the review queue.

I will compare the models directly with the Week-4 baseline before deciding whether the additional model complexity is justified.

In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=6,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

model_scores = {}

for name, model in models.items():
    model.fit(X_train_imp, y_train)

    model_scores[name] = model.predict_proba(
        X_test_imp
    )[:, 1]

    print(f"{name} trained.")

print("All models trained.")

Logistic Regression trained.
Decision Tree trained.
Random Forest trained.
All models trained.


In [18]:
print("Models available:")
print(list(model_scores.keys()))

for name, scores in model_scores.items():
    print(name, "→", len(scores), "test predictions")

Models available:
['Logistic Regression', 'Decision Tree', 'Random Forest']
Logistic Regression → 55887 test predictions
Decision Tree → 55887 test predictions
Random Forest → 55887 test predictions


In [19]:
from sklearn.metrics import average_precision_score, roc_auc_score

def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))

    top_k = np.argsort(scores)[::-1][:k]

    return y_true[top_k].mean()


def evaluate_model(name, y_true, scores):
    return {
        "Model": name,
        "Average Precision": average_precision_score(
            y_true,
            scores
        ),
        "ROC AUC": roc_auc_score(
            y_true,
            scores
        ),
        "Precision@10": precision_at_k(
            y_true,
            scores,
            10
        ),
        "Precision@50": precision_at_k(
            y_true,
            scores,
            50
        ),
        "Precision@100": precision_at_k(
            y_true,
            scores,
            100
        )
    }

In [20]:
model_results = pd.DataFrame([
    evaluate_model(
        name,
        y_test,
        scores
    )
    for name, scores in model_scores.items()
])

model_results.sort_values(
    "Precision@50",
    ascending=False
)

,Model,Average Precision,ROC AUC,Precision@10,Precision@50,Precision@100
2,Random Forest,0.880301,0.844464,1.0,0.98,0.96
1,Decision Tree,0.874933,0.841293,1.0,0.96,0.96
0,Logistic Regression,0.874937,0.841181,1.0,0.92,0.93


In [26]:


baseline_scores = X_test["impressions_feb"].fillna(0).to_numpy()

baseline_results = evaluate_model(
    "Simple Baseline",
    y_test,
    baseline_scores
)

print("Baseline results:")
print(baseline_results)

Baseline results:
{'Model': 'Simple Baseline', 'Average Precision': 0.4816908938167109, 'ROC AUC': 0.21027344058282696, 'Precision@10': np.float64(0.1), 'Precision@50': np.float64(0.02), 'Precision@100': np.float64(0.01)}


In [27]:
# STEP — Model vs baseline comparison

comparison_results = pd.concat(
    [
        pd.DataFrame([baseline_results]),
        model_results
    ],
    ignore_index=True
)

comparison_results = comparison_results.sort_values(
    "Average Precision",
    ascending=False
).reset_index(drop=True)

comparison_results

,Model,Average Precision,ROC AUC,Precision@10,Precision@50,Precision@100
0,Random Forest,0.880301,0.844464,1.0,0.98,0.96
1,Logistic Regression,0.874937,0.841181,1.0,0.92,0.93
2,Decision Tree,0.874933,0.841293,1.0,0.96,0.96
3,Simple Baseline,0.481691,0.210273,0.1,0.02,0.01


## Errors and interpretation

The Random Forest is used for the error analysis because it had the strongest measured ranking performance among the tested models.

I inspect false positives and false negatives to understand where the ranking model can fail. I also use permutation importance to identify which February signals are most influential for the model.

These results are used as decision-support evidence for prioritizing content review. They do not establish that the model will cause future performance improvements.

In [28]:


best_model_name = "Random Forest"
best_model = models[best_model_name]
best_scores = model_scores[best_model_name]

error_df = model_frame.loc[
    X_test.index,
    [
        "client_hash_id",
        "content_hash_id",
        "impressions_feb",
        "clicks_feb",
        "ctr_feb",
        "avg_position_feb",
        "engagement_rate_feb",
        "went_dark"
    ]
].copy()

error_df["predicted_score"] = best_scores

# Highest-scored false positives:
# predicted high risk, but did not go dark
false_positives = error_df[
    error_df["went_dark"] == 0
].sort_values(
    "predicted_score",
    ascending=False
).head(10)

# Lowest-scored false negatives:
# actually went dark, but received a relatively low risk score
false_negatives = error_df[
    error_df["went_dark"] == 1
].sort_values(
    "predicted_score",
    ascending=True
).head(10
)

print("Top false positives:")
display(false_positives)

print("\nTop false negatives:")
display(false_negatives)

Top false positives:


,client_hash_id,content_hash_id,impressions_feb,clicks_feb,ctr_feb,avg_position_feb,engagement_rate_feb,went_dark,predicted_score
84607,client_a80fca3f171ed1de,content_40e3112f8dfaa7ba,6.0,0.0,0.0,71.333333,NaN,0,0.939048
115483,client_f623b01661d4bfe4,content_70f32e2d50afcc43,17.0,0.0,0.0,90.705882,NaN,0,0.938420
21637,client_f623b01661d4bfe4,content_08fc6aa0b504f70b,13.0,0.0,0.0,82.230769,0.0,0,0.938266
31728,client_f623b01661d4bfe4,content_46ad753b44ed3c58,4.0,0.0,0.0,65.500000,NaN,0,0.938157
12916,client_a80fca3f171ed1de,content_257935b0475f10a1,4.0,0.0,0.0,61.250000,NaN,0,0.936444
4180,client_f623b01661d4bfe4,content_faa5751dea48f998,3.0,0.0,0.0,80.333333,0.0,0,0.932981
3080,client_f623b01661d4bfe4,content_fafa6357426cd829,12.0,0.0,0.0,63.250000,NaN,0,0.931536
116674,client_a80fca3f171ed1de,content_575832888c91f255,1.0,0.0,0.0,87.000000,NaN,0,0.931153
65604,client_f623b01661d4bfe4,content_8e934cf3d4bd5ce1,3.0,0.0,0.0,58.333333,NaN,0,0.931039
128412,client_2094c6eb080311d5,content_ad22711e724516ab,6.0,0.0,0.0,57.166667,NaN,0,0.930189



Top false negatives:


,client_hash_id,content_hash_id,impressions_feb,clicks_feb,ctr_feb,avg_position_feb,engagement_rate_feb,went_dark,predicted_score
52453,client_73cda7b4e4f265ea,content_a7f5da93ba6fda13,2692.0,8.0,0.002972,5.263744,NaN,1,0.006300
155026,client_73cda7b4e4f265ea,content_1fb678fd8f43c3af,3341.0,7.0,0.002095,4.363065,NaN,1,0.007916
37690,client_73cda7b4e4f265ea,content_835af1106517866a,4125.0,8.0,0.001939,1.938667,NaN,1,0.008498
119790,client_73cda7b4e4f265ea,content_92d2c3fc8bf462c9,17311.0,10.0,0.000578,5.094391,NaN,1,0.008513
144225,client_73cda7b4e4f265ea,content_892737fa57a46052,1361.0,7.0,0.005143,7.040411,NaN,1,0.009405
41841,client_73cda7b4e4f265ea,content_efa0b93f73308430,2237.0,7.0,0.003129,11.559231,NaN,1,0.009963
101685,client_73cda7b4e4f265ea,content_df1c14f140da47e1,2751.0,6.0,0.002181,4.125045,NaN,1,0.010978
118801,client_73cda7b4e4f265ea,content_154dce08305d7b2b,1269.0,7.0,0.005516,1.813239,NaN,1,0.011300
48473,client_73cda7b4e4f265ea,content_d42846150eef8c70,5770.0,7.0,0.001213,8.320104,NaN,1,0.011845
113248,client_73cda7b4e4f265ea,content_0e4050a52fe36b79,1857.0,7.0,0.003770,3.552504,NaN,1,0.011858


In [29]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    best_model,
    X_test_imp,
    y_test,
    scoring="average_precision",
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": perm.importances_mean,
    "Std": perm.importances_std
}).sort_values(
    "Importance",
    ascending=False
)

print("Permutation importance:")
display(importance_df)

Permutation importance:


,Feature,Importance,Std
0,impressions_feb,0.079231,0.001067
1,clicks_feb,0.018777,0.000900
2,ctr_feb,0.008149,0.000740
3,avg_position_feb,0.005228,0.000891
4,engagement_rate_feb,0.000331,0.000105


In [30]:
# Top-risk content summary

for k in [10, 50, 100, 500]:
    top_k = np.argsort(best_scores)[::-1][:k]
    observed_rate = y_test.iloc[top_k].mean()

    print(
        f"Top {k}: "
        f"{observed_rate:.3f} went-dark rate"
    )

Top 10: 1.000 went-dark rate
Top 50: 0.980 went-dark rate
Top 100: 0.960 went-dark rate
Top 500: 0.952 went-dark rate


The Random Forest produced the strongest measured ranking performance among the tested models. Its top-ranked test items had very high observed went-dark rates: 100% for the top 10, 98% for the top 50, 96% for the top 100, and 95.2% for the top 500. This indicates that the model is effective at prioritizing content with the observed March outcome in this held-out test sample. The false-positive and false-negative review shows where the ranking can still fail, while permutation importance helps identify which February signals contribute most to the predictions. These findings are directional and should be treated as decision-support evidence rather than evidence of causal impact.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.